# nb_setup_silver_metadata
Creates `meta_columns` from the logic in the original Databricks transform notebooks:
column selection (anything not listed is dropped), renames, casts, and value transforms.
Re-running **overwrites** the table, so keep manual edits in this notebook, not in the table.

In [1]:
from pyspark.sql import functions as F

# (source_expr, target_col, data_type, transform)
#  - source_expr: a bronze column name, or a Spark SQL expression (e.g. nested fields)
#  - transform:   initcap | upper | lower | trim | None
#  - a bronze column that is not listed here is dropped (e.g. url)
AUDIT = [("ingestion_timestamp", "ingestion_timestamp", "timestamp", None),
         ("source_file",         "source_file",         "string",    None)]

RESULT_LIKE = [
    ("season",        "season",              "int",    None),
    ("round",         "round",               "int",    None),
    ("constructorId", "constructor_id",      "string", None),
    ("driverId",      "driver_id",           "string", None),
    ("date",          "race_date",           "date",   None),
    ("raceName",      "race_name",           "string", "initcap"),
    ("grid",          "grid_position",       "int",    None),
    ("laps",          "completed_laps",      "int",    None),
    ("number",        "car_number",          "int",    None),
    ("points",        "points",              "float",  None),
    ("position",      "final_position",      "int",    None),
    ("positionText",  "final_position_text", "string", None),
    ("status",        "status",              "string", None),
] + AUDIT

spec = {
    "circuits": [
        ("circuitId",   "circuit_id",   "string", None),
        ("circuitName", "circuit_name", "string", "initcap"),
        ("lat",         "latitude",     "double", None),
        ("long",        "longitude",    "double", None),
        ("locality",    "locality",     "string", "initcap"),
        ("country",     "country",      "string", None),
    ] + AUDIT,
    "races": [
        ("season",    "season",    "int",    None),
        ("round",     "round",     "int",    None),
        ("raceName",  "race_name", "string", "initcap"),
        ("date",      "race_date", "date",   None),
        ("circuitId", "circuit_id","string", None),
    ] + AUDIT,
    "constructors": [
        ("constructorId", "constructor_id",   "string", None),
        ("name",          "constructor_name", "string", "initcap"),
        ("nationality",   "nationality",      "string", "initcap"),
    ] + AUDIT,
    "drivers": [
        ("driverId",    "driver_id",     "string", None),
        ("concat_ws(' ', name.givenName, name.familyName)", "driver_name", "string", "initcap"),
        ("dateOfBirth", "date_of_birth", "date",   None),
        ("nationality", "nationality",   "string", "initcap"),
    ] + AUDIT,
    "results": RESULT_LIKE,
    "sprints": RESULT_LIKE,
}

rows = [(e, i, src, tgt, dt, tf, 1)
        for e, items in spec.items()
        for i, (src, tgt, dt, tf) in enumerate(items, start=1)]

schema = ("entity string, ordinal int, source_expr string, target_col string, "
          "data_type string, transform string, is_active int")

(spark.createDataFrame(rows, schema)
      .write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable("meta_columns"))

display(spark.table("meta_columns").orderBy("entity", "ordinal"))

StatementMeta(, 1d7a5e35-ac53-451a-bd79-53501002acb4, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1d53567d-d007-49d1-929e-346e10e6e843)